In [ ]:
quart["ChangeMCFTR"] = 0
quart["ChangeRUCBTRNS"] = 0
quart["ChangeIZO"] = 0
quart["ChangeXAU"] = 0
quart["WeightMCFTR"] = 0
quart["WeightRUCBTRNS"] = 0
quart["WeightIZO"] = 0
quart["WeightXAU"] = 0
quart["IndexNO"] = 0

# делаем месяцы
is_new_month = quart["tradedate"].dt.month != quart["tradedate"].shift(1).dt.month
is_quarter_month = quart["tradedate"].dt.month.isin([1, 4, 7, 10])
quarter_trigger = is_new_month & is_quarter_month
quart["Marker"] = np.where(quarter_trigger, 1, 0)
quart = quart.reset_index()
quart = quart.drop(columns=["index"])

# Добавляем первую сртроку чтобы от нее дальше отталкиваться. Эти значения нельзя менять
starting_row = pd.Series(
    {
        "CloseMCFTR": 5885.95,
        "CloseRUCBTRNS": 146.04,
        "tradedate": pd.to_datetime("31.05.2023", dayfirst=True),
        "CloseIZO": 8093.218475,
        "CloseXAU": 159182.3098,
        "ChangeMCFTR": 0.0069990966699971,
        "ChangeRUCBTRNS": -0.000175496753310078,
        "ChangeIZO": -0.000685477547824997,
        "ChangeXAU": 0.00114893553887185,
        "WeightMCFTR": 34.1189237509017,
        "WeightRUCBTRNS": 30.6304832446833,
        "WeightIZO": 48.5950961330808,
        "WeightXAU": 12.5342641701075,
        "IndexNO": 125.878767298773,
        "Marker": 0,
    }
)
quart.loc[-1] = starting_row
quart = quart.sort_index()
quart = quart.reset_index(drop=True)
quart["ChangeMCFTR"] = quart["CloseMCFTR"].pct_change()
quart["ChangeRUCBTRNS"] = quart["CloseRUCBTRNS"].pct_change()
quart["ChangeIZO"] = quart["CloseIZO"].pct_change()
quart["ChangeXAU"] = quart["CloseXAU"].pct_change()
quart.iloc[0, 5] = 0.0069990966699971  # добавляем значения в ChangeMCFTR
quart.iloc[0, 6] = -0.000175496753310078  # добавляем значения в ChangeRUCBTRNS
quart.iloc[0, 7] = -0.000685477547824997
quart.iloc[0, 8] = 0.00114893553887185

# фцнкция которая считает индекс
def calculate_weights_INO(df):
    for i in range(1, len(df)):
        if df.loc[i, "Marker"] != 1:
            df.loc[i, "WeightMCFTR"] = (1 + df.loc[i, "ChangeMCFTR"]) * df.loc[
                i - 1, "WeightMCFTR"
            ]
            df.loc[i, "WeightRUCBTRNS"] = (
                1 + df.loc[i, "ChangeRUCBTRNS"]
            ) * df.loc[i - 1, "WeightRUCBTRNS"]
            df.loc[i, "WeightIZO"] = (1 + df.loc[i, "ChangeIZO"]) * df.loc[
                i - 1, "WeightIZO"
            ]
            df.loc[i, "WeightXAU"] = (1 + df.loc[i, "ChangeXAU"]) * df.loc[
                i - 1, "WeightXAU"
            ]
        else:
            df.loc[i, "WeightMCFTR"] = (
                df.loc[i - 1, "IndexNO"] * 0.25 * (1 + df.loc[i, "ChangeMCFTR"])
            )
            df.loc[i, "WeightRUCBTRNS"] = (
                df.loc[i - 1, "IndexNO"] * 0.25 * (1 + df.loc[i, "ChangeRUCBTRNS"])
            )
            df.loc[i, "WeightIZO"] = (
                df.loc[i - 1, "IndexNO"] * 0.4 * (1 + df.loc[i, "ChangeIZO"])
            )
            df.loc[i, "WeightXAU"] = (
                df.loc[i - 1, "IndexNO"] * 0.1 * (1 + df.loc[i, "ChangeXAU"])
            )
        df.loc[i, "IndexNO"] = (
            df.loc[i, "WeightMCFTR"]
            + df.loc[i, "WeightRUCBTRNS"]
            + df.loc[i, "WeightIZO"]
            + df.loc[i, "WeightXAU"]
        )
    return df

quart = calculate_weights_INO(quart)  # разница в среднем менее 0.01%
